# Мин. Обучение модели

Метод Дениса (dense) в минимальном виде

Устанавливаем наш пакет и импортируем все нужное

In [ ]:
%pip install -q git+https://github.com/tereshkovyv/transformer_lab.git

Подключаем Гугл диск, чтобы хранить там временные файлы

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/transformer_lab/data/wikitext-103"

Настраиваем конфигурацию

In [ ]:
SEED = 44
cfg = {"vocab_size": 32000, "context_length": 2048, "emb_dim": 512, "n_heads": 8,
       "n_layers": 12, "hidden_dim": 1536, "drop_rate": 0.0, "qkv_bias": False}
hp = {"max_length": 512, "batch_size": 8, "train_steps": 2048, "lr": 3e-4, "min_lr": 3e-5,
      "warmup_steps": 102, "weight_decay": 0.1, "betas": (0.9, 0.95), "eval_freq": 250, "eval_iter": 2}

Создаем объекты данных, модели, оптимайзера

In [ ]:
import torch
from torch.utils.data import DataLoader, RandomSampler
from transformer_lab.model.gpt_model import GPTModelV2
from transformer_lab.training.data import TokenDataset

device = "cuda"
torch.manual_seed(SEED)

train_ds = TokenDataset("train", hp["max_length"], DATA_DIR)
val_ds = TokenDataset("validation", hp["max_length"], DATA_DIR)
sampler = RandomSampler(train_ds, num_samples=hp["train_steps"] * hp["batch_size"])
train_loader = DataLoader(train_ds, batch_size=hp["batch_size"], sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=hp["batch_size"])

model = GPTModelV2(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"],
                              weight_decay=hp["weight_decay"], betas=hp["betas"])

Обучаем

In [ ]:
from transformer_lab.training.train import train_model_min
train_losses, val_losses, lrs = train_model_min(model, train_loader, val_loader, optimizer, device,
                                             warmup_steps=hp["warmup_steps"], min_lr=hp["min_lr"],
                                             eval_freq=hp["eval_freq"], eval_iter=hp["eval_iter"])

Сохраняем веса

In [ ]:
from google.colab import files
torch.save(model.state_dict(), "model.pth")
files.download("model.pth")

Рисуем кривые обучения

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, len(train_losses) + 1), train_losses, alpha=0.5, label="train (current step)")
axes[0].plot(*zip(*val_losses), marker="o", label=f"validation ({hp['eval_iter'] * hp['batch_size']} sequences)")
axes[0].set(title="Loss", xlabel="Step", ylabel="Loss")
axes[0].legend()
axes[1].plot(range(1, len(lrs) + 1), lrs)
axes[1].set(title="Learning rate", xlabel="Step", ylabel="lr")
plt.show()